# VR-1 Digital Twin Summary

This notebook explains how to use the coupled VR-1 digital twin files, what the generated images show, and how the physics maps onto the code.

The files in this folder are meant to be used together:
- `digital_twin.py`: static ROM for reactivity and flux shape
- `digital_twin_driver.py`: coupled ROM + PRKE driver
- `visualize_thermal_power.py`: animation of localized thermal power across macro time steps
- `visualize_short_kinetics.py`: short-time kinetics animation that highlights prompt and delayed neutron effects
- `visualize_thermal_widget.ipynb`: interactive frame-by-frame viewer for thermal power
- generated outputs: `coupled_digital_twin_summary.png`, `thermal_power_animation.gif`, `short_kinetics_animation.gif`

## How to Run It

Run the coupled model first, then the visualization scripts. The recommended environment is `vr1-openmc`.

```bash
cd /home/username/VR1-openmc/digital_twin_files
/home/username/miniconda3/envs/vr1-openmc/bin/python digital_twin_driver.py
/home/username/miniconda3/envs/vr1-openmc/bin/python visualize_thermal_power.py --results coupled_digital_twin_results.npz --out thermal_power_animation.gif --z-index 35 --interval-ms 300
/home/username/miniconda3/envs/vr1-openmc/bin/python visualize_short_kinetics.py --out short_kinetics_animation.gif --t-final 0.2 --dt 0.0001 --rho-step 0.003 --frame-stride 25
```

OR

```
conda activate vr1-openmc
python file.py --options

If you want the interactive view, open `visualize_thermal_widget.ipynb` in Jupyter or VS Code and run its cells.

## What the Images Show

### `coupled_digital_twin_summary.png`
A compact snapshot of the coupled simulation. It shows: 
- macro-step reactivity from the ROM
- neutron density from the PRKE solver on a log scale
- normalized total core power on a log scale
- a final localized thermal power slice through the reactor core

### `thermal_power_animation.gif`
An animation of the thermal localized power slice as the macro state changes. This is useful for seeing how the spatial power field moves when the ROM updates the core configuration.

### `short_kinetics_animation.gif`
A short-time kinetics animation that isolates the PRKE response to a reactivity step. This is the best image for seeing prompt-neutron behavior and the slower delayed-neutron contribution.

## How the Physics Maps to the Code

The coupled model uses an adiabatic approximation. That means the spatial flux shape is updated on a slower macro time step, while the reactor kinetics are advanced on a finer micro time step.

### ROM side
`digital_twin.py` loads the precomputed OpenMC training data, fits a surrogate for k-effective, and projects flux snapshots onto a reduced basis. When you pass in rod heights and water-density multipliers, it returns:
- reactivity, computed from the surrogate k-effective
- a flattened flux vector reconstructed from the reduced basis

### PRKE side
`digital_twin_driver.py` uses the point-kinetics equations to advance neutron density `P(t)`. The prompt term responds quickly to reactivity changes, while the delayed-neutron precursor groups act more slowly and smooth the transient.

### Coupling logic
At each macro step, the ROM provides a new reactivity value and flux shape. Inside that macro interval, the PRKE solver is stepped many times with interpolated reactivity. The localized power is then formed by multiplying the instantaneous `P(t)` by the spatial flux vector.

This is why the short kinetics GIF shows a fast prompt response followed by a slower delayed-neutron tail, while the thermal-power animation shows how the spatial field changes across macro steps.

## Quick Guide For The Images

- `coupled_digital_twin_summary.png`: watch the log-scale neutron density and power traces. The fast rise near the start is the prompt-neutron response, while the slower tail reflects delayed neutrons.
- `thermal_power_animation.gif`: watch how the spatial thermal power slice changes from macro step to macro step. This is the ROM side of the model, where rod motion and water-density changes reshape the flux field.
- `short_kinetics_animation.gif`: watch the short-time PRKE response. This is the best image for seeing the prompt jump and the delayed-neutron evolution on a fine time scale.

A good way to read the set is to compare them together: the summary plot shows the overall time history, the thermal GIF shows the spatial power map, and the short kinetics GIF isolates the time-dependent neutron population physics.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

base = Path('.')
images = [
    base / 'coupled_digital_twin_summary.png',
    base / 'thermal_power_animation.gif',
    base / 'short_kinetics_animation.gif',
]

for image_path in images:
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f'Missing: {image_path}')